# 08 · Experiment tracking, artifacts, SQLite, MySQL, and MSSQL

Build a local registry, query its normalized records, attach artifacts, and compare SQL dialects.

In [ ]:
# Local/VS Code and Colab bootstrap. No secret or API key is used.
import json
import os
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    repo = Path("/content/ai-rookie-skill-tree-lab")
    if not repo.exists():
        subprocess.run(
            ["git", "clone", "https://github.com/chelsietao/ai-rookie-skill-tree-lab.git", str(repo)],
            check=True,
        )
    os.chdir(repo)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[full,dev]"], check=True)
elif Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)

ROOT = Path.cwd().resolve()
DATA_DIR = ROOT / "data/generated/ci"
METRICS_DIR = ROOT / "artifacts/metrics/ci"
RUN_TRAINING = False  # Set True when you want to regenerate the chapter's metrics.
assert (ROOT / "pyproject.toml").exists(), "Open the repository root or notebooks directory."
print({"environment": "Colab" if IN_COLAB else "local/VS Code", "root": str(ROOT)})

## Why tracking is part of modeling

A metric without its seed, data version, configuration, environment, and artifact cannot be audited.
This project keeps the registry local and transparent so its schema can be inspected directly.

In [ ]:
import time
from rookie_lab.tracking.metadata import enrich_metrics
if not (DATA_DIR / "manifest.json").exists():
    subprocess.run([sys.executable, "scripts/prepare_data.py", "--profile", "ci"], check=True)
sample = enrich_metrics({"roc_auc": 0.71}, lab="notebook", profile="ci",
                        data_dir=DATA_DIR, started=time.perf_counter(),
                        config_path="configs/experiments/quick.yaml")
display(pd.Series(sample))

## Create an isolated SQLite registry

In [ ]:
import tempfile
from rookie_lab.tracking.registry import ExperimentRegistry, ExperimentRun
registry_path = Path(tempfile.mkdtemp()) / "registry.sqlite"
registry = ExperimentRegistry(registry_path)
registry.log(ExperimentRun("nb-001", "classical", "sklearn", 20260820,
                           {"model": "logistic"}, {"roc_auc": 0.71}))
registry.log_artifact("nb-001", "figure", "artifacts/figures/classical.png", "demo-sha")
print(registry.list(), registry.artifacts("nb-001"))

## Query the physical schema

The API wraps the registry, but SQL remains useful for auditing row shape and query plans.

In [ ]:
import sqlite3
with sqlite3.connect(registry_path) as connection:
    display(pd.read_sql_query("SELECT run_id, lab, framework, seed FROM experiment_runs", connection))
    display(pd.read_sql_query("SELECT * FROM artifacts", connection))

## Primary keys, foreign keys, and indexes

The retail schema separates entities and events. Foreign keys protect customer/product references;
indexes accelerate common lookup paths. They do not replace application validation or monitoring.

In [ ]:
schemas = {name: Path(f"sql/{name}/schema.sql").read_text()
           for name in ("sqlite", "mysql", "mssql")}
for name, sql in schemas.items():
    print(name, {"tables": sql.upper().count("CREATE TABLE"),
                 "indexes": sql.upper().count("CREATE INDEX")})

## Dialect differences

In [ ]:
from rookie_lab.database.dialects import compiled_schemas
compiled = compiled_schemas()
display(pd.DataFrame({name: {"DDL preview": ddl[:120]} for name, ddl in compiled.items()}).T)

## Security boundary

Passwords are read from environment variables at runtime and are never embedded in a URL example,
notebook, committed config, or test fixture. SQLite is the fully exercised local path; MySQL has a
service CI smoke test; MSSQL is schema-validated but requires an external service for integration.

In [ ]:
print(Path(".env.example").read_text())

## Exercises

Add a unique constraint that prevents duplicate view events; compare an `EXPLAIN` plan before and
after an index; decide whether an artifact should be deleted when its run is removed; design an
append-only alternative to `INSERT OR REPLACE`.